In [ ]:
!pip install ucimlrepo xgboost scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split#2_dataset k dui bhage bhag korsi
from sklearn.preprocessing import StandardScaler#2_normalize kore same scale e anar jnno
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score #3_shob evolution metrics load korsi and eder kaaj holo actual datar sathe predict kora datar diff koddur seta ber kora..
from sklearn.ensemble import RandomForestRegressor#2_Rndm forest import
import xgboost as xgb#2_XGboosting import
import tensorflow as tf#3_ei line er maddhome google er deep learnning library ta import kortesi
from tensorflow.keras.models import Sequential #tensorflow er keras module er sequential architecture ta import krtesi-eta ektar por ekta layer shajate help korbe
from tensorflow.keras.layers import Dense, Dropout #3_NN er fully connected layer and overfitting tackle er jnno dropout


#1_start
steel_data = fetch_ucirepo(id=851)
df = pd.concat([steel_data.data.features, steel_data.data.targets], axis=1)

df.columns = df.columns.str.strip()

target_col = [c for c in df.columns if 'usage' in c.lower() or 'kwh' in c.lower()]
target_col = target_col[0] if target_col else df.columns[-1]

date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
if date_cols:
    d_col = date_cols[0]
    df[d_col] = pd.to_datetime(df[d_col], errors='coerce')
    df['hour'] = df[d_col].dt.hour.fillna(0)
    df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24)
    df = df.drop(columns=[d_col, 'hour'])

cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

df = df.dropna()

X_clean = df.drop(columns=[target_col])
y_clean = df[target_col]




#2_START
X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

scaler = StandardScaler() #making stndrd scaler object
X_train_scaled = scaler.fit_transform(X_train) #scaling trainning features
X_test_scaled = scaler.transform(X_test) #scaling testing features


#making randomforest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train) #randomforest train kora
rf_preds = rf_model.predict(X_test) #random forest test using test features

#making xgboosting regression model
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)#xgboost train kora
xgb_preds = xgb_model.predict(X_test)#xg prediction using test data


#2_END


#3_start
#ANN structure
ann_model = Sequential([  #sequential neural network
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)), #shape- etar maddhome dataset e input feature(column) koita ase seta auto count kore
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1) #4th dense layer/output layer
])

ann_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

#Train ANN
ann_model.fit(X_train_scaled, y_train, validation_split=0.2, epochs=25, batch_size=64, verbose=0)
ann_preds = ann_model.predict(X_test_scaled).flatten() #1.Prediction using test data and Flatten


results = {}  #empty result dictoinary to store R^2


#Evaluate function to calculate the preformance of every model
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    results[name] = r2 * 100 #r2 percentage convertion
    print(f"{name}:")
    print(f"RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2 Score: {r2*100:.2f}%\n")

print("="*40)
print("         MODEL EVALUATION RESULTS        ")
print("="*40)
evaluate(y_test, rf_preds, "Random Forest")
evaluate(y_test, xgb_preds, "XGBoost")
evaluate(y_test, ann_preds, "Deep Neural Network (ANN)")


#graph
plt.figure(figsize=(8, 5))
sns.barplot(x=list(results.keys()), y=list(results.values()), palette='viridis')
plt.ylabel('R2 Score (%)')
plt.title('Model Comparison - R2 Score Performance')
plt.ylim(80, 100)
for i, v in enumerate(results.values()):
    plt.text(i, v + 0.5, f"{v:.2f}%", ha='center', fontweight='bold')
plt.show()

In [ ]:
import os
import joblib
import numpy as np
import time
import subprocess
from flask import Flask, render_template, request
from sklearn.ensemble import RandomForestRegressor

# 1. Retrain model briefly and save
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

joblib.dump(rf_model, 'steel_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

# 2. Create HTML UI Template
os.makedirs('templates', exist_ok=True)
html_code = """
<!DOCTYPE html>
<html>
<head>
    <title>Steel Energy Consumption Predictor</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #eef2f5; margin: 0; padding: 40px; }
        .container { max-width: 500px; background: white; padding: 30px; border-radius: 12px; box-shadow: 0px 4px 15px rgba(0,0,0,0.1); margin: auto; }
        h2 { text-align: center; color: #1a365d; margin-bottom: 20px; }
        label { font-weight: 600; color: #2d3748; margin-top: 12px; display: block; font-size: 14px; }
        input { width: 100%; padding: 10px; margin-top: 5px; border: 1px solid #cbd5e0; border-radius: 6px; box-sizing: border-box; }
        button { width: 100%; background-color: #2b6cb0; color: white; padding: 12px; border: none; border-radius: 6px; margin-top: 25px; font-size: 16px; font-weight: bold; cursor: pointer; }
        button:hover { background-color: #2c5282; }
        .result { margin-top: 25px; padding: 15px; background: #ebf8ff; border: 1px solid #bee3f8; border-radius: 8px; text-align: center; font-size: 18px; color: #2b6cb0; font-weight: bold; }
    </style>
</head>
<body>
    <div class="container">
        <h2>Steel Industry Energy Predictor</h2>
        <form action="/predict" method="post">
            <label>Lagging Current Reactive Power (kVARh):</label>
            <input type="number" step="any" name="lag_react" required value="{{ inputs.lag_react if inputs else '35' }}">

            <label>Leading Current Reactive Power (kVARh):</label>
            <input type="number" step="any" name="lead_react" required value="{{ inputs.lead_react if inputs else '0' }}">

            <label>CO2 Emissions (ppm):</label>
            <input type="number" step="any" name="co2" required value="{{ inputs.co2 if inputs else '0.02' }}">

            <label>Lagging Current Power Factor (%):</label>
            <input type="number" step="any" name="lag_pf" required value="{{ inputs.lag_pf if inputs else '88' }}">

            <label>Leading Current Power Factor (%):</label>
            <input type="number" step="any" name="lead_pf" required value="{{ inputs.lead_pf if inputs else '100' }}">

            <button type="submit">Predict Energy Usage</button>
        </form>

        {% if prediction %}
            <div class="result">
                Predicted Energy Usage: {{ prediction }} kWh
            </div>
        {% endif %}
    </div>
</body>
</html>
"""

with open('templates/index.html', 'w') as f:
    f.write(html_code)

# 3. Setup Cloudflare Tunnel (No Passwords / No Login Pages)
os.system("pkill cloudflared")
if not os.path.exists("cloudflared"):
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
    os.system("chmod +x cloudflared")

os.system("./cloudflared tunnel --url http://localhost:5000 > cloudflare.log 2>&1 &")
time.sleep(8)

if os.path.exists('cloudflare.log'):
    with open('cloudflare.log', 'r') as f:
        log = f.read()
        for line in log.split('\n'):
            if 'https://' in line and 'trycloudflare.com' in line:
                # Extract full URL accurately
                parts = line.split()
                for part in parts:
                    if 'trycloudflare.com' in part and 'https://' in part:
                        clean_url = part.strip()
                        print("==================================================")
                        print("Click this Public URL to open UI:", clean_url)
                        print("==================================================")
                        break

# 4. Start Flask Server
app = Flask(__name__)
model = joblib.load('steel_model.pkl')

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    if request.method == 'POST':
        lag_react = float(request.form['lag_react'])
        lead_react = float(request.form['lead_react'])
        co2 = float(request.form['co2'])
        lag_pf = float(request.form['lag_pf'])
        lead_pf = float(request.form['lead_pf'])

        inputs = {
            'lag_react': request.form['lag_react'],
            'lead_react': request.form['lead_react'],
            'co2': request.form['co2'],
            'lag_pf': request.form['lag_pf'],
            'lead_pf': request.form['lead_pf']
        }

        feature_count = X_train.shape[1]
        input_data = np.zeros(feature_count)

        input_data[0] = lag_react
        input_data[1] = lead_react
        input_data[2] = co2
        input_data[3] = lag_pf
        input_data[4] = lead_pf

        pred = model.predict([input_data])[0]

        return render_template('index.html', prediction=f"{pred:.2f}", inputs=inputs)

if __name__ == '__main__':
    app.run(port=5000)

Click this Public URL to open UI: https://congratulations-scope-that-jason.trycloudflare.com
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/Sep/2026 06:28:55] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Sep/2026 06:28:56] "GET /favicon.ico HTTP/1.1" 404 -
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [07/Sep/2026 06:28:58] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Sep/2026 06:29:55] "GET / HTTP/1.1" 200 -
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [07/Sep/2026 06:29:58] "POST /predict HTTP/1.1" 200 -
/usr/